# Human 3PTB workflow: installed preparation, search and evaluation

This bounded human workflow consumes public MolSysMT and DockingMT tools. The accepted 3PTB recipe retains the complete explicit state, source maps, charge/type reports and submitted bytes. It is a software/integration control, not an affinity estimate or validation of the preparation model.

**Stop boundary:** this notebook ends before visualization. We will discuss and plan visualization with the maintainer before implementing or running it.

The original 24 searches / 167 poses and their positional evaluation stay immutable. A separate receiver adds optional declared-state symmetry measurements. One new search is registered in advance: rigid ligand, native box, seed 7, exhaustiveness 1, CPU 1, at most 9 poses. It tests the installed end-to-end route; no favorable outcome or similarity to historical stochastic output is required.

Run outside the source checkout using the verified receiving kernel. Set `DOCKINGMT_HUMAN_DATA_ROOT` to the source-free harness containing `devtools/` and `devguide/`; set `DOCKINGMT_HUMAN_OUTPUT` to a **new** output directory and `DOCKINGMT_EXPECTED_PREFIX` to the receiving prefix. The output directory is never overwritten.

In [1]:
import hashlib
import importlib.metadata as metadata
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import molsysmt as msm
import numpy as np
import pyunitwizard as puw
import dockingmt as dmt

root = Path(os.environ['DOCKINGMT_HUMAN_DATA_ROOT']).resolve()
prefix = Path(os.environ['DOCKINGMT_EXPECTED_PREFIX']).resolve()
output = Path(os.environ['DOCKINGMT_HUMAN_OUTPUT']).resolve()
assert Path(dmt.__file__).resolve().is_relative_to(prefix)
assert not (root / 'dockingmt').exists()
assert not (root / 'molsysviewer_dockingmt').exists()
output.mkdir(parents=True, exist_ok=False)
sys.path.insert(0, str(root))  # Case recipe/data only; no scientific package sources.

from devtools.qualify_3ptb_admission import admit
from devtools.qualify_3ptb_search import evaluate, execution_protocol

versions = {name: metadata.version(name) for name in
            ['dockingmt', 'molsysmt', 'argdigest', 'pyunitwizard', 'vina', 'rdkit']}
print(json.dumps({'versions': versions, 'dockingmt': dmt.__file__,
                  'molsysmt': msm.__file__}, indent=2))

{
  "versions": {
    "dockingmt": "0.0.0+180.g1decee5",
    "molsysmt": "0.23.0+78.gc35252d89",
    "argdigest": "0.15.0+5.g3d1ec53",
    "pyunitwizard": "0.28.1",
    "vina": "1.2.7",
    "rdkit": "2026.3.1"
  },
  "dockingmt": "/tmp/dockingmt-human-20261020/receiving/lib/python3.14/site-packages/dockingmt/__init__.py",
  "molsysmt": "/tmp/dockingmt-human-20261020/environment/lib/python3.14/site-packages/molsysmt/__init__.py"
}


## 1. Read the original compressed source through MolSysMT

`get_missing_bonds=False` preserves the source's declared connectivity. Compressed transport and PDB interpretation belong to MolSysMT. The original entry contains 1,701 atoms, 287 groups, one structure and 21 explicitly declared bonds.

In [2]:
source_path = root / 'devguide/validation/data/3ptb_prospective/3PTB.pdb.gz'
source_sha256 = hashlib.sha256(source_path.read_bytes()).hexdigest()
source = msm.convert(source_path, to_form='molsysmt.MolSys', get_missing_bonds=False)
counts = msm.get(source, n_atoms=True, n_groups=True, n_structures=True, n_bonds=True,
                 output_type='dictionary')
assert counts == {'n_atoms': 1701, 'n_groups': 287, 'n_structures': 1, 'n_bonds': 21}
print(counts)

{'n_atoms': 1701, 'n_groups': 287, 'n_structures': 1, 'n_bonds': 21}


## 2. Apply the already registered preparation recipe

Inspect `devguide/validation/3ptb_prospective_protocol.md` and the case-specific public-API orchestration in `devtools/qualify_3ptb_admission.py`. Reusing that audited recipe keeps notebook cells focused on scientific choices; it does not introduce a molecular reader, matcher, charge model or geometry routine here.

The declared hypothesis uses the dry, rigid protein, HID57/HIE40/HIE91, six deposited disulfides, charged termini and ligand benzamidinium +1 (`NC(=[NH2+])c1ccccc1`). Waters/calcium are excluded. Missing H, chemistry, Gasteiger–Marsili charges and named AutoDock4 types are supplied by explicit MolSysMT operations. Protonation is a declared hypothesis, not an inference from the experiment. Attribute-loss and generated-H identity decisions are retained in the recipe report. No environment refinement is claimed.

In [3]:
receptor, partners, observed_reference, boxes, admission, chemical_context = admit(
    source_path, include_chemical_context=True)
print(json.dumps({'source': admission['source_sha256'],
                  'rigid_atoms': partners['rigid'].n_atoms,
                  'flexible_atoms': partners['flexible'].n_atoms,
                  'native_box': admission['boxes']['native'],
                  'ligand_formal_charge': 1}, indent=2))
(output / 'admission.json').write_text(json.dumps(admission, indent=2, allow_nan=False))

{
  "source": "288f7954d4d013fa8eab3808e1037e2958e2dcd12c9eca65a3f1014404a9f9a2",
  "rigid_atoms": 13,
  "flexible_atoms": 13,
  "native_box": {
    "unrounded": {
      "center": [
        -1.7591111111111113,
        14.461000000000002,
        16.915777777777777
      ],
      "lengths": [
        19.999999999999996,
        19.999999999999996,
        19.999999999999996
      ],
      "size": [
        19.999999999999996,
        19.999999999999996,
        19.999999999999996
      ],
      "unit": "angstrom"
    },
    "backend_projection": {
      "center": [
        -1.759111,
        14.461,
        16.915778
      ],
      "size": [
        20.0,
        20.0,
        20.0
      ],
      "unit": "angstrom"
    },
    "native_heavy_containment": [
      true,
      true,
      true,
      true,
      true,
      true,
      true,
      true,
      true
    ],
    "submitted": false
  },
  "ligand_formal_charge": 1
}


97038061

## 3. Execute the one predeclared installed-route search

The native 20 × 20 × 20 angstrom box is centered on the nine observed heavy atoms. The selected protocol is the first pre-existing registered row; no parameter is chosen after inspecting outcomes. Fresh stochastic output need not be byte-identical to the historical producer. Capture the actual submitted receptor/ligand bytes and native output.

If a prior attempt already produced the registered result before a later analysis failure, set `DOCKINGMT_HUMAN_RESTORE_FROM` to that exact saved result. The explicit recovery path verifies protocol and submitted preparation hashes, then restores it without another native search. Its receipt records zero new searches and the original source path.

In [4]:
row = admission['population'][0]
assert (row['arm'], row['domain'], row['protocol']['parameters']['seed'],
        row['protocol']['parameters']['exhaustiveness']) == ('rigid', 'native', 7, 1)
protocol = execution_protocol(row)
problem = dmt.DockingProblem(receptor, partners['rigid'], boxes['native'],
    metadata={'case': '3PTB', 'purpose': 'installed_human_workflow_control'})
restore_from = os.environ.get('DOCKINGMT_HUMAN_RESTORE_FROM')
if restore_from:
    original_result_path = Path(restore_from).resolve()
    result = dmt.DockingResult.from_dict(json.loads(original_result_path.read_text()))
    assert result.protocol_info == protocol.to_dict()
    for role, prepared in [('receptor', receptor), ('partner', partners['rigid'])]:
        assert result.provenance['backend_artifacts'][role]['sha256'] == hashlib.sha256(
            prepared.to_pdbqt().encode()).hexdigest()
    new_native_searches = 0
else:
    result = dmt.dock(problem, protocol)
    new_native_searches = 1
dmt.verify_captured_inputs(result.provenance['backend_artifacts'])
result_record = result.to_dict()
result_path = output / 'new_result.json'
result_path.write_text(json.dumps(result_record, indent=2, allow_nan=False))
print({'returned_poses': len(result.poses), 'new_native_searches': new_native_searches,
       'result_path': str(result_path)})

{'returned_poses': 5, 'new_native_searches': 0, 'result_path': '/tmp/dockingmt-human-20261020/human-recovered-output/new_result.json'}


## 4. Evaluate, preserving positional and symmetry criteria separately

The metric population is the original nine heavy atoms; the complete 18-atom declared ligand supplies chemical context. MolSysMT evaluates positional RMSD in the shared receptor frame, without fitting or periodic imaging. Its symmetry tool preserves the exact declared formal charge, bond order, stereo and H representation; it does not normalize the two amidinium N roles by resonance.

The original positional report remains authoritative for its registered criterion. `symmetry_evaluation` is a separate optional measurement with its maps, graph digests, software, bibliography and top-N interpretation. Unselected H coordinates supply no measured pose geometry. An unsupported graph or search-limit failure stops the evaluation rather than manufacturing an equivalent match.

Template application leaves the nine observed atoms without stereo metadata. For this specific benzamidinium hypothesis, explicitly declare `atom_stereochemistry="unspecified"` on a detached context using the public MolSysMT setter. This is not a local chirality classifier or inference from coordinates. MolSysMT independently rejects any potential stereo left unassigned; the two N roles remain distinct. The original preparation and saved result are unchanged.

In [5]:
# Explicit case declaration, not a stereo assignment or a geometry inference.
# MolSysMT still rejects any potentially stereogenic site left unassigned.
stereo_context = msm.copy(chemical_context)
msm.set(stereo_context, atom_stereochemistry='unspecified')
options = {'molecular_system': stereo_context, 'chemical_state': 'structure',
           'reference_chemical_state': 'structure'}
evaluation = evaluate(result, stereo_context, boxes['native'], admission, 'rigid',
                      symmetry_options=options)
report = evaluation['report']
print(json.dumps({'positional_top_n': report['top_n'],
                  'symmetry_top_n': report['symmetry_evaluation']['top_n'],
                  'poses': report['symmetry_evaluation']['poses']}, indent=2))
(output / 'new_evaluation.json').write_text(json.dumps(evaluation, indent=2, allow_nan=False))

{
  "positional_top_n": [
    {
      "requested": 1,
      "considered": 1,
      "recovered": true
    },
    {
      "requested": 9,
      "considered": 5,
      "recovered": true
    }
  ],
  "symmetry_top_n": [
    {
      "requested": 1,
      "considered": 1,
      "recovered": true
    },
    {
      "requested": 9,
      "considered": 5,
      "recovered": true
    }
  ],
  "poses": [
    {
      "position": 1,
      "pose_id": "pose_1",
      "rank": 1,
      "raw_rmsd": 1.9336974082484215,
      "rmsd": 1.090716940569111,
      "recovered": true,
      "correspondence_index": 1,
      "n_near_optimal_correspondences": 1
    },
    {
      "position": 2,
      "pose_id": "pose_2",
      "rank": 2,
      "raw_rmsd": 0.22834574175529807,
      "rmsd": 0.22834574175529807,
      "recovered": true,
      "correspondence_index": 0,
      "n_near_optimal_correspondences": 1
    },
    {
      "position": 3,
      "pose_id": "pose_3",
      "rank": 3,
      "raw_rmsd": 9.99874749933

1537154

## 5. Restore the saved result without another search

The saved record must retain coordinates, units, scores, identities and captured inputs. Recovery calls no native docking operation. The new receiving evaluation under pm/fs must agree numerically with the earlier evaluation in explicit angstrom.

In [6]:
restored = dmt.DockingResult.from_dict(json.loads(result_path.read_text()))
assert restored.to_dict() == result_record
with puw.context(standard_units=['pm', 'fs', 'K', 'mole', 'dalton', 'e', 'kJ/mol', 'degrees']):
    received = evaluate(restored, stereo_context, boxes['native'], admission, 'rigid',
                        symmetry_options=options)
np.testing.assert_allclose(
    [p['rmsd'] for p in received['report']['symmetry_evaluation']['poses']],
    [p['rmsd'] for p in report['symmetry_evaluation']['poses']], rtol=0, atol=1e-10)
assert received['complete_energy_vectors'] == evaluation['complete_energy_vectors']
assert received['containment'] == evaluation['containment']
print('Saved-result recovery passed without another native search.')

Saved-result recovery passed without another native search.


## 6. Receive all historical searches as a new, separately identified measurement

The immutable original search archive retains its 24 reporting failures; the later positional receiver retains the corrected 24/167 evaluation. This cell neither overwrites nor reruns that population. It adds independent symmetry measurements using the explicitly declared chemical hypothesis and compares original nine-heavy-atom positional values, scores and containment with the retained receiving record.

In [7]:
import gzip
search_path = root / 'devguide/validation/data/3ptb_search/audit_2026-10-10.json.gz'
old_path = search_path.with_name('evaluation_2026-10-10.json.gz')
assert hashlib.sha256(search_path.read_bytes()).hexdigest() == '97104903c93b3f7a729a5a6ac196cdb7b59d686e318efdcf324f70131abba1f2'
assert hashlib.sha256(old_path.read_bytes()).hexdigest() == '8351d4db25b2ad25de413158a8d3dc0a5e4fee31678ea06649994a1832fa53af'
searches = json.loads(gzip.decompress(search_path.read_bytes()))
previous = json.loads(gzip.decompress(old_path.read_bytes()))
received_runs = []
for saved, old in zip(searches['runs'], previous['runs'], strict=True):
    historical = dmt.DockingResult.from_dict(saved['result'])
    box = dmt.BoxRegion(puw.quantity(saved['backend_box']['center'], 'angstrom'),
                        puw.quantity(saved['backend_box']['size'], 'angstrom'))
    measured = evaluate(historical, stereo_context, box, admission, saved['arm'],
                        symmetry_options=options)
    np.testing.assert_allclose([p['rmsd'] for p in measured['report']['poses']],
        [p['rmsd'] for p in old['evaluation']['report']['poses']], rtol=0, atol=1e-10)
    assert measured['containment'] == old['evaluation']['containment']
    assert measured['complete_energy_vectors'] == old['evaluation']['complete_energy_vectors']
    received_runs.append({'arm': saved['arm'], 'domain': saved['domain'],
                         'seed': saved['protocol']['parameters']['seed'],
                         'exhaustiveness': saved['protocol']['parameters']['exhaustiveness'],
                         'evaluation': measured})
assert len(received_runs) == 24
assert sum(r['evaluation']['report']['n_poses'] for r in received_runs) == 167
(output / 'saved_symmetry_receiving.json').write_text(json.dumps(received_runs, allow_nan=False))
print({'historical_runs_received': 24, 'saved_poses_received': 167,
       'additional_searches_in_receiving': 0})

{'historical_runs_received': 24, 'saved_poses_received': 167, 'additional_searches_in_receiving': 0}


## Stop here: visualization requires discussion and planning

No viewer, browser scene or rendering is invoked. Discuss the intended scientific view, receptor/reference/pose correspondence, domain display and interaction needs before continuing. Successful execution above qualifies this bounded installed workflow, not biological activity, broad docking accuracy, public package admission or visualization.

In [8]:
origins = {}
for name, module in tuple(sys.modules.items()):
    if name == 'dockingmt' or name.startswith('dockingmt.'):
        path = getattr(module, '__file__', None)
        if path:
            assert Path(path).resolve().is_relative_to(prefix), (name, path)
            origins[name] = path
receipt = {'finished_utc': datetime.now(timezone.utc).isoformat(),
    'versions': versions, 'source_sha256': source_sha256,
    'imports': origins, 'new_native_searches': new_native_searches,
    'restored_result_source': restore_from,
    'stereo_declaration': 'unspecified for this declared benzamidinium context; provider validates potential stereo',
    'historical_runs_received': 24, 'historical_poses_received': 167,
    'visualization_executed': False,
    'output_files': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                     for p in sorted(output.iterdir()) if p.is_file()}}
(output / 'receipt.json').write_text(json.dumps(receipt, indent=2, allow_nan=False))
print('Stopped before visualization.')

Stopped before visualization.
